# HuBERT Layer-wise Analysis for Native Language Identification

This notebook analyzes which HuBERT layers best capture accent-related information for Native Language Identification.

In [ ]:
import torch
import torchaudio
from transformers import Wav2Vec2Processor, HubertModel
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.svm import SVC
from sklearn.model_selection import cross_val_score
from sklearn.metrics import accuracy_score
import librosa

# Configuration
HUBERT_MODEL = "facebook/hubert-base-ls960"
SAMPLE_RATE = 16000

In [ ]:
# Load HuBERT model
processor = Wav2Vec2Processor.from_pretrained(HUBERT_MODEL)
model = HubertModel.from_pretrained(HUBERT_MODEL)

In [ ]:
def load_audio(file_path, target_sr=16000):
    """
    Load and preprocess audio file
    """
    waveform, sr = torchaudio.load(file_path)
    
    # Resample if needed
    if sr != target_sr:
        resampler = torchaudio.transforms.Resample(sr, target_sr)
        waveform = resampler(waveform)
    
    # Convert to mono if stereo
    if waveform.shape[0] > 1:
        waveform = torch.mean(waveform, dim=0, keepdim=True)
        
    return waveform.squeeze()

In [ ]:
def extract_layer_representations(waveform, model, processor):
    """
    Extract representations from all HuBERT layers
    """
    # Process audio
    inputs = processor(waveform, sampling_rate=SAMPLE_RATE, return_tensors="pt", padding="longest")
    
    # Extract features from all layers
    with torch.no_grad():
        outputs = model(**inputs, output_hidden_states=True)
        hidden_states = outputs.hidden_states  # Tuple of tensors for each layer
    
    return hidden_states

In [ ]:
def evaluate_layer_performance(hidden_states, labels):
    """
    Evaluate classification performance for each layer representation
    """
    layer_accuracies = []
    
    for i, layer_features in enumerate(hidden_states):
        # Average pool over time dimension
        pooled_features = layer_features.mean(dim=1).numpy()
        
        # Simple SVM classifier
        clf = SVC(kernel='rbf')
        
        # Cross-validation
        scores = cross_val_score(clf, pooled_features, labels, cv=5)
        mean_accuracy = scores.mean()
        
        layer_accuracies.append(mean_accuracy)
        print(f"Layer {i}: Accuracy = {mean_accuracy:.4f}")
    
    return layer_accuracies

In [ ]:
# Placeholder for actual data loading
# In practice, you would load the IndicAccentDb dataset here

# Example usage (uncomment and adapt when you have actual data):
"""
# Load your dataset
# audio_paths = [...]  # List of audio file paths
# labels = [...]       # Corresponding language labels

# Extract features from one sample to demonstrate
# waveform = load_audio(audio_paths[0])
# hidden_states = extract_layer_representations(waveform, model, processor)

# Evaluate all layers
# layer_accuracies = evaluate_layer_performance(hidden_states, labels)

# Plot results
# plt.figure(figsize=(12, 6))
# plt.plot(layer_accuracies, marker='o')
# plt.title('HuBERT Layer Performance for NLI')
# plt.xlabel('Layer')
# plt.ylabel('Classification Accuracy')
# plt.grid(True)
# plt.show()
"""

## Analysis Results

After running the analysis on the complete dataset, you would observe which layers provide the best accent discrimination. Typically, middle layers of self-supervised models like HuBERT tend to capture the most linguistically relevant information.